# Mosaic — Reproject and Stitch 4 Tiles (36JTM, 36JUM, 36JTN, 36JUN)

Combines the four verified tiles into a single aligned raster grid for each data source: Sentinel-2 (optical bands + indices), Sentinel-1 (VV, VH), DEM, and the consolidated UNOSAT flood label.

**Project:**  Development and Evaluation of a Deep Learning Model for Flood Detection and Drought Prediction Using Satellite Remote Sensing Data in South Africa
**Study Area:** KwaZulu-Natal, South Africa
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Reference Product:** UNOSAT FL20220418ZAF
**Student:** Athindothe Valencia Marubini
**Student No:** 219160643
**Supervisor:** Prof. IE Davidson
**Co-Supervisor:** Dr O.P Babalola
**Institution:** Cape Peninsula University of Technology (CPUT)


Sentinel-2 tiles were all acquired in EPSG:32736 (UTM Zone 36S). Sentinel-1, exported via Earth Engine using each tile's own footprint as the export region, got auto-assigned a UTM zone per tile based on each region's centroid — 36JUM and 36JUN came back in EPSG:32736, while 36JTM and 36JTN came back in EPSG:32735 (UTM Zone 35S), since their footprints sit closer to the 35S/36S zone boundary (30°E). All Sentinel-1 tiles get reprojected to EPSG:32736 before mosaicking, to match Sentinel-2, DEM, and the label.

Inputs:
- Sentinel-2: Sentinel2/Raw/*.SAFE/ (4 tiles x 2 dates)
- Sentinel-1: Sentinel1_v2/<tile>/*.tif (4 tiles x 2 dates)
- DEM: DEM/Processed/DEM_4tile_30m_native.tif
- Label: Labels_v2/UNOSAT_flood_extent_consolidated.geojson

Output: one aligned, mosaicked raster per source, all in EPSG:32736, 10m resolution, covering the full 4-tile extent.

---
## Step 1: Mount Drive and Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install rasterio geopandas --quiet

In [ ]:
import os
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
import numpy as np
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

ROOT = '/content/drive/MyDrive/KZN_Research_Colab/'
MOSAIC_DIR = ROOT + 'Mosaic_4tile/'
os.makedirs(MOSAIC_DIR, exist_ok=True)

TARGET_CRS = 'EPSG:32736'
TILES = ['36JTM', '36JUM', '36JTN', '36JUN']
PRE_DATE  = '20220329'
POST_DATE = '20220428'

print('Target CRS:', TARGET_CRS)
print('Tiles:', TILES)
print('Output directory:', MOSAIC_DIR)

Target CRS: EPSG:32736
Tiles: ['36JTM', '36JUM', '36JTN', '36JUN']
Output directory: /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/


---
## Step 2: Reproject Sentinel-1 Tiles to Consistent CRS

Tiles already in EPSG:32736 are copied as-is. Tiles in EPSG:32735
(36JTM, 36JTN) are reprojected to EPSG:32736 using bilinear resampling,
appropriate for continuous SAR backscatter values.

In [ ]:
def reproject_raster(src_path, dst_path, target_crs, resampling=Resampling.bilinear):
    with rasterio.open(src_path) as src:
        if str(src.crs) == target_crs:
            # Already correct CRS - just copy through unchanged
            import shutil
            shutil.copy2(src_path, dst_path)
            return False  # no reprojection needed

        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })
        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=resampling
                )
        return True  # reprojection was performed

S1_REPROJ_DIR = MOSAIC_DIR + 'S1_reprojected/'
os.makedirs(S1_REPROJ_DIR, exist_ok=True)

s1_files = {
    ('36JTM', 'pre'):  ROOT + 'Sentinel1_v2/36JTM/S1_36JTM_PreFlood_20220324.tif',
    ('36JTM', 'post'): ROOT + 'Sentinel1_v2/36JTM/S1_36JTM_PostFlood_20220422.tif',
    ('36JUM', 'pre'):  ROOT + 'Sentinel1_v2/36JUM/S1_36JUM_PreFlood_20220324.tif',
    ('36JUM', 'post'): ROOT + 'Sentinel1_v2/36JUM/S1_36JUM_PostFlood_20220424.tif',
    ('36JTN', 'pre'):  ROOT + 'Sentinel1_v2/36JTN/S1_36JTN_PreFlood_20220324.tif',
    ('36JTN', 'post'): ROOT + 'Sentinel1_v2/36JTN/S1_36JTN_PostFlood_20220422.tif',
    ('36JUN', 'pre'):  ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PreFlood_20220324.tif',
    ('36JUN', 'post'): ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PostFlood_20220424.tif',
}

s1_reprojected_paths = {}
print('Reprojecting Sentinel-1 tiles to', TARGET_CRS, '...\n')
for (tile, period), src_path in s1_files.items():
    dst_path = S1_REPROJ_DIR + f'S1_{tile}_{period}.tif'
    if not os.path.exists(src_path):
        print(f'  [MISSING] {tile} {period}: {src_path}')
        continue
    was_reprojected = reproject_raster(src_path, dst_path, TARGET_CRS)
    s1_reprojected_paths[(tile, period)] = dst_path
    action = 'REPROJECTED' if was_reprojected else 'copied (already correct CRS)'
    print(f'  [{action}] {tile} {period}')

print('\nDone.')

Reprojecting Sentinel-1 tiles to EPSG:32736 ...

  [copied (already correct CRS)] 36JTM pre
  [REPROJECTED] 36JTM post
  [copied (already correct CRS)] 36JUM pre
  [copied (already correct CRS)] 36JUM post
  [copied (already correct CRS)] 36JTN pre
  [REPROJECTED] 36JTN post
  [copied (already correct CRS)] 36JUN pre
  [copied (already correct CRS)] 36JUN post

Done.


---
## Step 3: Verify Reprojection



In [ ]:
print('Post-reprojection CRS check:\n')
all_consistent = True
for (tile, period), path in s1_reprojected_paths.items():
    with rasterio.open(path) as src:
        match = str(src.crs) == TARGET_CRS
        if not match:
            all_consistent = False
        print(f'  {tile} {period}: CRS={src.crs}  [{"OK" if match else "MISMATCH"}]')

print()
print('All Sentinel-1 tiles consistent:' , all_consistent)

Post-reprojection CRS check:

  36JTM pre: CRS=EPSG:32736  [OK]
  36JTM post: CRS=EPSG:32736  [OK]
  36JUM pre: CRS=EPSG:32736  [OK]
  36JUM post: CRS=EPSG:32736  [OK]
  36JTN pre: CRS=EPSG:32736  [OK]
  36JTN post: CRS=EPSG:32736  [OK]
  36JUN pre: CRS=EPSG:32736  [OK]
  36JUN post: CRS=EPSG:32736  [OK]

All Sentinel-1 tiles consistent: True


---
## Step 4: Mosaic Sentinel-1 (Pre-flood and Post-flood Separately)

Tiles are merged using rasterio.merge, which handles overlapping
regions and produces one continuous raster per acquisition date.

In [ ]:
import rasterio

s1_check_paths = {
    '36JTM': '/content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_reprojected/S1_36JTM_pre.tif',
    '36JUM': '/content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_reprojected/S1_36JUM_pre.tif',
    '36JTN': '/content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_reprojected/S1_36JTN_pre.tif',
    '36JUN': '/content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_reprojected/S1_36JUN_pre.tif',
}

print('Checking each reprojected S1 tile individually:\n')
for tile, path in s1_check_paths.items():
    with rasterio.open(path) as src:
        print(f'{tile}: shape={src.shape}  bounds={src.bounds}  dtype={src.dtypes[0]}')

Checking each reprojected S1 tile individually:

36JTM: shape=(10981, 10981)  bounds=BoundingBox(left=199965.37729354633, bottom=6590203.670582946, right=309775.37729354633, top=6700013.670582946)  dtype=float64
36JUM: shape=(10981, 10981)  bounds=BoundingBox(left=299985.37729354633, bottom=6590203.670582946, right=409795.37729354633, top=6700013.670582946)  dtype=float64
36JTN: shape=(10981, 10981)  bounds=BoundingBox(left=199965.37729354633, bottom=6690223.670582946, right=309775.37729354633, top=6800033.670582946)  dtype=float64
36JUN: shape=(10981, 10981)  bounds=BoundingBox(left=299985.37729354633, bottom=6690223.670582946, right=409795.37729354633, top=6800033.670582946)  dtype=float64


In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
import shutil
import os

def reproject_raster(src_path, dst_path, target_crs, resampling=Resampling.bilinear, target_dtype='float32'):
    with rasterio.open(src_path) as src:
        if str(src.crs) == target_crs and src.dtypes[0] == target_dtype:
            shutil.copy2(src_path, dst_path)
            return False
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs, 'transform': transform,
            'width': width, 'height': height, 'dtype': target_dtype
        })
        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i), destination=rasterio.band(dst, i),
                    src_transform=src.transform, src_crs=src.crs,
                    dst_transform=transform, dst_crs=target_crs, resampling=resampling
                )
        return True

# Re-run reprojection for ALL tiles (even ones already in the right CRS),
# this time forcing float32 dtype to cut memory use in half
S1_REPROJ_DIR = MOSAIC_DIR + 'S1_reprojected/'

s1_files = {
    ('36JTM', 'pre'):  ROOT + 'Sentinel1_v2/36JTM/S1_36JTM_PreFlood_20220324.tif',
    ('36JTM', 'post'): ROOT + 'Sentinel1_v2/36JTM/S1_36JTM_PostFlood_20220422.tif',
    ('36JUM', 'pre'):  ROOT + 'Sentinel1_v2/36JUM/S1_36JUM_PreFlood_20220324.tif',
    ('36JUM', 'post'): ROOT + 'Sentinel1_v2/36JUM/S1_36JUM_PostFlood_20220424.tif',
    ('36JTN', 'pre'):  ROOT + 'Sentinel1_v2/36JTN/S1_36JTN_PreFlood_20220324.tif',
    ('36JTN', 'post'): ROOT + 'Sentinel1_v2/36JTN/S1_36JTN_PostFlood_20220422.tif',
    ('36JUN', 'pre'):  ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PreFlood_20220324.tif',
    ('36JUN', 'post'): ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PostFlood_20220424.tif',
}

s1_reprojected_paths = {}
print('Re-reprojecting all S1 tiles to float32 +', TARGET_CRS, '...\n')
for (tile, period), src_path in s1_files.items():
    dst_path = S1_REPROJ_DIR + f'S1_{tile}_{period}.tif'
    if os.path.exists(dst_path):
        os.remove(dst_path)  # remove the old float64 version first
    reproject_raster(src_path, dst_path, TARGET_CRS, target_dtype='float32')
    s1_reprojected_paths[(tile, period)] = dst_path
    with rasterio.open(dst_path) as check:
        print(f'  {tile} {period}: dtype={check.dtypes[0]}  shape={check.shape}')

Re-reprojecting all S1 tiles to float32 + EPSG:32736 ...

  36JTM pre: dtype=float32  shape=(10981, 10981)
  36JTM post: dtype=float32  shape=(12147, 12147)
  36JUM pre: dtype=float32  shape=(10981, 10981)
  36JUM post: dtype=float32  shape=(10981, 10981)
  36JTN pre: dtype=float32  shape=(10981, 10981)
  36JTN post: dtype=float32  shape=(12116, 12117)
  36JUN pre: dtype=float32  shape=(10981, 10981)
  36JUN post: dtype=float32  shape=(10981, 10981)


In [ ]:
def mosaic_rasters_lowmem(file_paths, output_path):
    srcs = [rasterio.open(p) for p in file_paths]
    merge(srcs, dst_path=output_path)  # writes directly to disk
    for s in srcs:
        s.close()
    return output_path

for period in ['pre', 'post']:
    paths = [s1_reprojected_paths[(tile, period)] for tile in TILES if (tile, period) in s1_reprojected_paths]
    out_path = MOSAIC_DIR + f'S1_mosaic_{period}.tif'
    if os.path.exists(out_path):
        os.remove(out_path)
    mosaic_rasters_lowmem(paths, out_path)
    with rasterio.open(out_path) as src:
        print(f'S1 {period}-flood mosaic created: {out_path}')
        print(f'  Shape: {src.shape}  Bands: {src.count}  dtype: {src.dtypes[0]}  CRS: {src.crs}')
    print()

S1 pre-flood mosaic created: /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_mosaic_pre.tif
  Shape: (20983, 20983)  Bands: 2  dtype: float32  CRS: EPSG:32736

S1 post-flood mosaic created: /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/S1_mosaic_post.tif
  Shape: (22139, 21576)  Bands: 2  dtype: float32  CRS: EPSG:32736



---
## Step 5: Mosaic Sentinel-2 (Pre-flood and Post-flood, per Required Band)

Sentinel-2 bands are read directly from each .SAFE granule's IMG_DATA
folder. Only the bands used in the original 12-channel stack are
processed: B02, B03, B04, B08 (10m), B11, B12 (20m, resampled to 10m
during mosaic).

In [ ]:
def find_s2_band_file(safe_dir, band):
    """Find a specific band's .jp2 file, preferring 10m resolution where available."""
    candidates = []
    for dirpath, dirnames, filenames in os.walk(safe_dir):
        for f in filenames:
            if f.endswith('.jp2') and f'_{band}_' in f:
                candidates.append(os.path.join(dirpath, f))
    if not candidates:
        return None
    # Prefer 10m over 20m/60m if multiple resolutions exist
    for res in ['10m', '20m', '60m']:
        for c in candidates:
            if res in c:
                return c
    return candidates[0]

S2_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

s2_safe_dirs = {
    ('36JTM', 'pre'):  ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220329T073609_N0510_R092_T36JTM_20240523T094629.SAFE/',
    ('36JTM', 'post'): ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220428T073609_N0510_R092_T36JTM_20240606T065344.SAFE/',
    ('36JUM', 'pre'):  ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220329T073609_N0510_R092_T36JUM_20240523T094629.SAFE/',
    ('36JUM', 'post'): ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220428T073609_N0510_R092_T36JUM_20240606T065344.SAFE/',
    ('36JTN', 'pre'):  ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220329T073609_N0510_R092_T36JTN_20240523T094629.SAFE/',
    ('36JTN', 'post'): ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220428T073609_N0510_R092_T36JTN_20240606T065344.SAFE/',
    ('36JUN', 'pre'):  ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220329T073609_N0510_R092_T36JUN_20240523T094629.SAFE/',
    ('36JUN', 'post'): ROOT + 'Sentinel2/Raw/S2B_MSIL2A_20220428T073609_N0510_R092_T36JUN_20240606T065344.SAFE/',
}

print('Locating S2 band files (this confirms paths before mosaicking)...\n')
s2_band_paths = {}
for (tile, period), safe_dir in s2_safe_dirs.items():
    for band in S2_BANDS:
        path = find_s2_band_file(safe_dir, band)
        if path is None:
            print(f'  [MISSING] {tile} {period} {band}')
        s2_band_paths[(tile, period, band)] = path

print('Band lookup complete.')

Locating S2 band files (this confirms paths before mosaicking)...

Band lookup complete.


In [ ]:
S2_MOSAIC_DIR = MOSAIC_DIR + 'S2_bands/'
os.makedirs(S2_MOSAIC_DIR, exist_ok=True)

for period in ['pre', 'post']:
    for band in S2_BANDS:
        paths = [s2_band_paths[(tile, period, band)] for tile in TILES
                 if s2_band_paths.get((tile, period, band)) is not None]
        if len(paths) != len(TILES):
            print(f'  [SKIP] {period} {band}: only {len(paths)}/{len(TILES)} tiles found')
            continue
        out_path = S2_MOSAIC_DIR + f'S2_{band}_{period}.tif'
        if os.path.exists(out_path):
            os.remove(out_path)
        mosaic_rasters_lowmem(paths, out_path)
        with rasterio.open(out_path) as src:
            print(f'  [OK] {period} {band}: {src.shape}  dtype={src.dtypes[0]}  CRS={src.crs}')

print('\nSentinel-2 band mosaicking complete.')

  [OK] pre B02: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] pre B03: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] pre B04: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] pre B08: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] pre B11: (10491, 10491)  dtype=uint16  CRS=EPSG:32736


...

  [OK] post B03: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] post B04: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] post B08: (20982, 20982)  dtype=uint16  CRS=EPSG:32736


  [OK] post B11: (10491, 10491)  dtype=uint16  CRS=EPSG:32736


  [OK] post B12: (10491, 10491)  dtype=uint16  CRS=EPSG:32736

Sentinel-2 band mosaicking complete.


---
## Step 6: Mosaic DEM

The DEM was already exported for the full 4-tile extent in one piece (see B2_acquisition.ipynb, DEM export step), so no mosaicking is required —
this step verifies it covers the same area as the S1/S2 mosaics rather
than assuming it does.

In [ ]:
DEM_PATH = ROOT + 'DEM/Processed/DEM_4tile_30m_native.tif'

with rasterio.open(DEM_PATH) as dem_src, rasterio.open(MOSAIC_DIR + 'S1_mosaic_pre.tif') as s1_src:
    print('DEM:')
    print(f'  CRS: {dem_src.crs}  Bounds: {dem_src.bounds}  Res: {dem_src.res}')
    print('S1 mosaic (for comparison):')
    print(f'  CRS: {s1_src.crs}  Bounds: {s1_src.bounds}  Res: {s1_src.res}')

    dem_covers = (dem_src.bounds.left <= s1_src.bounds.left and
                  dem_src.bounds.bottom <= s1_src.bounds.bottom and
                  dem_src.bounds.right >= s1_src.bounds.right and
                  dem_src.bounds.top >= s1_src.bounds.top)
    print()
    print('DEM fully covers S1/S2 mosaic extent:', dem_covers)
    if str(dem_src.crs) != TARGET_CRS:
        print(f'NOTE: DEM CRS ({dem_src.crs}) differs from target ({TARGET_CRS}) - will be reprojected during stack assembly.')

DEM:
  CRS: EPSG:4326  Bounds: BoundingBox(left=29.864996439340985, bottom=-30.816032905986653, right=32.07458254368977, top=-28.89238055657311)  Res: (0.0002694945852358564, 0.0002694945852358564)
S1 mosaic (for comparison):
  CRS: EPSG:32736  Bounds: BoundingBox(left=199965.37729354633, bottom=6590203.670582946, right=409795.37729354633, top=6800033.670582946)  Res: (10.0, 10.0)

DEM fully covers S1/S2 mosaic extent: False
NOTE: DEM CRS (EPSG:4326) differs from target (EPSG:32736) - will be reprojected during stack assembly.


In [ ]:
from rasterio.warp import transform_bounds

with rasterio.open(dem_path) as dem_src, rasterio.open(MOSAIC_DIR + 'S1_mosaic_pre.tif') as s1_src:
    # Reproject DEM's bounding box into the S1 mosaic's CRS for a fair comparison
    dem_bounds_in_target_crs = transform_bounds(
        dem_src.crs, s1_src.crs,
        dem_src.bounds.left, dem_src.bounds.bottom,
        dem_src.bounds.right, dem_src.bounds.top
    )

    print('DEM bounds (reprojected to EPSG:32736):', dem_bounds_in_target_crs)
    print('S1 mosaic bounds (native EPSG:32736)  :', s1_src.bounds)

    dem_left, dem_bottom, dem_right, dem_top = dem_bounds_in_target_crs
    dem_covers = (dem_left <= s1_src.bounds.left and
                  dem_bottom <= s1_src.bounds.bottom and
                  dem_right >= s1_src.bounds.right and
                  dem_top >= s1_src.bounds.top)
    print()
    print('DEM (reprojected) fully covers S1/S2 mosaic extent:', dem_covers)

DEM bounds (reprojected to EPSG:32736): (194253.20754589472, 6586579.768249158, 411483.05149732495, 6803585.50463157)
S1 mosaic bounds (native EPSG:32736)  : BoundingBox(left=199965.37729354633, bottom=6590203.670582946, right=409795.37729354633, top=6800033.670582946)

DEM (reprojected) fully covers S1/S2 mosaic extent: True


---
## Step 7: Rasterize the Consolidated UNOSAT Label to the Mosaic Grid

Converts the vector flood extent from B1_study_area_definition.ipynb
into a raster aligned exactly to the Sentinel-1/Sentinel-2 mosaic grid —
same CRS, resolution, and pixel alignment.

In [ ]:
from rasterio.features import rasterize

flood_gdf = gpd.read_file(ROOT + 'Labels_v2/UNOSAT_flood_extent_consolidated.geojson')
flood_gdf = flood_gdf.to_crs(TARGET_CRS)

with rasterio.open(MOSAIC_DIR + 'S1_mosaic_post.tif') as ref:
    out_shape = (ref.height, ref.width)
    transform = ref.transform
    meta = ref.meta.copy()

label_array = rasterize(
    [(geom, 1) for geom in flood_gdf.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype='uint8'
)

meta.update({'count': 1, 'dtype': 'uint8'})
label_out_path = MOSAIC_DIR + 'flood_label_mosaic.tif'
with rasterio.open(label_out_path, 'w', **meta) as dst:
    dst.write(label_array, 1)

flood_px = int((label_array == 1).sum())
total_px = label_array.size
print(f'Label rasterized: {label_out_path}')
print(f'  Shape: {label_array.shape}')
print(f'  Flood pixels: {flood_px:,}  ({100*flood_px/total_px:.4f}% of mosaic)')

Label rasterized: /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/flood_label_mosaic.tif
  Shape: (22139, 21576)
  Flood pixels: 98,715  (0.0207% of mosaic)


---
## Step 8: Summary

In [ ]:
print('=' * 65)
print('MOSAIC BUILD COMPLETE — SUMMARY')
print('=' * 65)
print(f'  Tiles combined   : {TILES}')
print(f'  Target CRS       : {TARGET_CRS}')
print(f'  Output directory : {MOSAIC_DIR}')
print()
print('  Outputs:')
print(f'    S1_mosaic_pre.tif')
print(f'    S1_mosaic_post.tif')
for band in S2_BANDS:
    print(f'    S2_bands/S2_{band}_pre.tif')
    print(f'    S2_bands/S2_{band}_post.tif')
print(f'    flood_label_mosaic.tif')
print()
print('Next step: assemble the 12-channel stack from these mosaicked')
print('bands (spectral indices, SAR pre/post, DEM) — same channel order')

MOSAIC BUILD COMPLETE — SUMMARY
  Tiles combined   : ['36JTM', '36JUM', '36JTN', '36JUN']
  Target CRS       : EPSG:32736
  Output directory : /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/

  Outputs:
    S1_mosaic_pre.tif
    S1_mosaic_post.tif
    S2_bands/S2_B02_pre.tif
    S2_bands/S2_B02_post.tif
    S2_bands/S2_B03_pre.tif
    S2_bands/S2_B03_post.tif
    S2_bands/S2_B04_pre.tif
    S2_bands/S2_B04_post.tif
    S2_bands/S2_B08_pre.tif
    S2_bands/S2_B08_post.tif
    S2_bands/S2_B11_pre.tif
    S2_bands/S2_B11_post.tif
    S2_bands/S2_B12_pre.tif
    S2_bands/S2_B12_post.tif
    flood_label_mosaic.tif

Next step: assemble the 12-channel stack from these mosaicked
bands (spectral indices, SAR pre/post, DEM) — same channel order


In [ ]:
# re running  S1 reprojection step, but only for post-flood, using the
# corrected files. Pre-flood S1

s1_post_corrected = {
    '36JTM': ROOT + 'Sentinel1_v2/36JTM/S1_36JTM_PostFlood_v2_20220417.tif',
    '36JUM': ROOT + 'Sentinel1_v2/36JUM/S1_36JUM_PostFlood_v2_20220417.tif',
    '36JTN': ROOT + 'Sentinel1_v2/36JTN/S1_36JTN_PostFlood_20220422.tif',  # original, accepted as-is
    '36JUN': ROOT + 'Sentinel1_v2/36JUN/S1_36JUN_PostFlood_v3_20220427.tif',
}

S1_REPROJ_DIR = MOSAIC_DIR + 'S1_reprojected/'
TARGET_CRS = 'EPSG:32736'

print('Re-reprojecting POST-FLOOD S1 tiles only (float32, correct CRS)...\n')
for tile, src_path in s1_post_corrected.items():
    dst_path = S1_REPROJ_DIR + f'S1_{tile}_post.tif'
    if os.path.exists(dst_path):
        os.remove(dst_path)
    reproject_raster(src_path, dst_path, TARGET_CRS, target_dtype='float32')
    with rasterio.open(dst_path) as check:
        print(f'  {tile}: dtype={check.dtypes[0]}  shape={check.shape}  crs={check.crs}')

Re-reprojecting POST-FLOOD S1 tiles only (float32, correct CRS)...

  36JTM: dtype=float32  shape=(10981, 10981)  crs=EPSG:32736
  36JUM: dtype=float32  shape=(10981, 10981)  crs=EPSG:32736
  36JTN: dtype=float32  shape=(12116, 12117)  crs=EPSG:32736
  36JUN: dtype=float32  shape=(10981, 10981)  crs=EPSG:32736


In [ ]:
out_path = MOSAIC_DIR + 'S1_mosaic_post.tif'
if os.path.exists(out_path):
    os.remove(out_path)

paths = [S1_REPROJ_DIR + f'S1_{tile}_post.tif' for tile in ['36JTM', '36JUM', '36JTN', '36JUN']]
mosaic_rasters_lowmem(paths, out_path)

with rasterio.open(out_path) as src:
    print(f'New S1 post-flood mosaic: {src.shape}  dtype={src.dtypes[0]}  CRS={src.crs}')

New S1 post-flood mosaic: (21537, 21547)  dtype=float32  CRS=EPSG:32736
